In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
#from torch.utils.data.dataset import _T_co
torch.manual_seed(0)
import os
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"


In [3]:
from transformers import DistilBertTokenizerFast,DistilBertForSequenceClassification

In [4]:
# device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")

In [5]:

device="cpu"

In [6]:
RANDOM_SEED = 0
NUM_EPOCHS = 3
#torch.manual_seed(RANDOM_SEED) # generator function

In [7]:
df = pd.read_csv('sample30.csv')

In [8]:
df

,id,brand,categories,manufacturer,name,reviews_date,reviews_didPurchase,reviews_doRecommend,reviews_rating,reviews_text,reviews_title,reviews_userCity,reviews_userProvince,reviews_username,user_sentiment
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",Universal Music Group / Cash Money,Pink Friday: Roman Reloaded Re-Up (w/dvd),2012-11-30T06:21:45.000Z,NaN,NaN,5,i love this album. it's very good. more to the...,Just Awesome,Los Angeles,NaN,joshua,Positive
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,2017-07-09T00:00:00.000Z,True,NaN,5,Good flavor. This review was collected as part...,Good,NaN,NaN,dorothy w,Positive
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,2017-07-09T00:00:00.000Z,True,NaN,5,Good flavor.,Good,NaN,NaN,dorothy w,Positive
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",K-Y,K-Y Love Sensuality Pleasure Gel,2016-01-06T00:00:00.000Z,False,False,1,I read through the reviews on here before look...,Disappointed,NaN,NaN,rebecca,Negative
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",K-Y,K-Y Love Sensuality Pleasure Gel,2016-12-21T00:00:00.000Z,False,False,1,My husband bought this gel for us. The gel cau...,Irritation,NaN,NaN,walker557,Negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-23T00:00:00.000Z,False,True,5,I got this conditioner with Influenster to try...,Softness!!,NaN,NaN,laurasnchz,Positive
29996,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-27T00:00:00.000Z,False,True,5,"I love it , I received this for review purpose...",I love it,NaN,NaN,scarlepadilla,Positive
29997,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-21T00:00:00.000Z,False,True,5,First of all I love the smell of this product....,Hair is so smooth after use,NaN,NaN,liviasuexo,Positive
29998,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-11T00:00:00.000Z,False,True,5,I received this through Influenster and will n...,Perfect for my oily hair!,NaN,NaN,ktreed95,Positive


In [9]:
#todo convert to graph nn
device = "cpu"

In [10]:
reviews = df[['id','brand','categories','reviews_text','reviews_title','reviews_rating','user_sentiment']]

In [11]:
reviews.dropna(inplace=True)

In [12]:
reviews

,id,brand,categories,reviews_text,reviews_title,reviews_rating,user_sentiment
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",i love this album. it's very good. more to the...,Just Awesome,5,Positive
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Good flavor. This review was collected as part...,Good,5,Positive
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Good flavor.,Good,5,Positive
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",I read through the reviews on here before look...,Disappointed,1,Negative
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",My husband bought this gel for us. The gel cau...,Irritation,1,Negative
...,...,...,...,...,...,...,...
29995,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",I got this conditioner with Influenster to try...,Softness!!,5,Positive
29996,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...","I love it , I received this for review purpose...",I love it,5,Positive
29997,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",First of all I love the smell of this product....,Hair is so smooth after use,5,Positive
29998,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",I received this through Influenster and will n...,Perfect for my oily hair!,5,Positive


## finetuning for sentiment classification

In [13]:
reviews['reviews']=reviews['reviews_title'].add(" ").add(reviews["reviews_text"])
reviews.drop(['reviews_title','reviews_text'], axis=1, inplace=True)

In [14]:
reviews

,id,brand,categories,reviews_rating,user_sentiment,reviews
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",5,Positive,Just Awesome i love this album. it's very good...
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",5,Positive,Good Good flavor. This review was collected as...
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",5,Positive,Good Good flavor.
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",1,Negative,Disappointed I read through the reviews on her...
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",1,Negative,Irritation My husband bought this gel for us. ...
...,...,...,...,...,...,...
29995,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",5,Positive,Softness!! I got this conditioner with Influen...
29996,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",5,Positive,"I love it I love it , I received this for revi..."
29997,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",5,Positive,Hair is so smooth after use First of all I lov...
29998,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",5,Positive,Perfect for my oily hair! I received this thro...


In [15]:
reviews['user_sentiment']=reviews['user_sentiment'].map({'Positive':1,'Negative':0})

In [16]:
reviews

,id,brand,categories,reviews_rating,user_sentiment,reviews
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",5,1,Just Awesome i love this album. it's very good...
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",5,1,Good Good flavor. This review was collected as...
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",5,1,Good Good flavor.
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",1,0,Disappointed I read through the reviews on her...
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",1,0,Irritation My husband bought this gel for us. ...
...,...,...,...,...,...,...
29995,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",5,1,Softness!! I got this conditioner with Influen...
29996,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",5,1,"I love it I love it , I received this for revi..."
29997,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",5,1,Hair is so smooth after use First of all I lov...
29998,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",5,1,Perfect for my oily hair! I received this thro...


In [18]:
reviews.to_csv('reviews.csv',index=False)

In [19]:
train_reviews = reviews.iloc[:len(reviews)//8]

In [20]:
train_reviews

,id,brand,categories,reviews_rating,user_sentiment,reviews
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",5,1,Just Awesome i love this album. it's very good...
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",5,1,Good Good flavor. This review was collected as...
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",5,1,Good Good flavor.
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",1,0,Disappointed I read through the reviews on her...
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",1,0,Irritation My husband bought this gel for us. ...
...,...,...,...,...,...,...
367,AV1l8zRZvKc47QAVhnAv,Olay,"Personal Care,Skin Care,Anti-Aging,Beauty,Face...",5,1,Wonderfully hydrating I have been using this c...
368,AV1l8zRZvKc47QAVhnAv,Olay,"Personal Care,Skin Care,Anti-Aging,Beauty,Face...",5,1,Hydration PLUS My skin was so dry dull before ...
369,AV1l8zRZvKc47QAVhnAv,Olay,"Personal Care,Skin Care,Anti-Aging,Beauty,Face...",5,1,The best hydration product I have used in year...
370,AV1l8zRZvKc47QAVhnAv,Olay,"Personal Care,Skin Care,Anti-Aging,Beauty,Face...",5,0,amazing product Ive been using Regenerist Deep...


In [21]:
validation_reviews = reviews.iloc[len(reviews)//80:len(reviews)//40]

In [22]:
test_reviews = reviews.iloc[len(reviews)//40:len(reviews)//40+len(reviews)//100]

In [23]:
len(test_reviews)

298

In [24]:
len(validation_reviews)

373

In [25]:
len(train_reviews)

372

In [26]:
model = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

In [27]:
from transformers import DistilBertTokenizerFast

# Load the standard, official DistilBERT tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

In [28]:
list(train_reviews['reviews'])[:2]

["Just Awesome i love this album. it's very good. more to the hip hop side than her current pop sound.. SO HYPE! i listen to this everyday at the gym! i give it 5star rating all the way. her metaphors are just crazy.",
 'Good Good flavor. This review was collected as part of a promotion.']

In [29]:
x=list(train_reviews['reviews'].values)[:2]


In [30]:


from transformers import PreTrainedModel

In [31]:
dir(PreTrainedModel)

['T_destination',
 '__annotate_func__',
 '__annotations__',
 '__annotations_cache__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_adjust_bias',
 '_adjust_missing_and_unexpected_keys',
 '_apply',
 '_auto_class',
 '_backward_compatibility_gradient_checkpointing',
 '_call_impl',
 '_can_compile_fullgraph',
 '_can_record_outputs',
 '_can_set_attn_implementation',
 '_can_set_experts_implementation',
 '_check_and_adjust_attn_implementation',
 '_check_and_adjust_experts_implementation',
 '_compatible_flash_implementations',
 '_compiled_call_impl',
 '_copy_lm_head_

In [32]:
train_encodings = tokenizer(list(train_reviews['reviews'].values),padding=True,truncation=True,return_tensors="pt").to(device)
valid_encodings =  tokenizer(list(validation_reviews['reviews'].values),padding=True,truncation=True,return_tensors="pt").to(device)
test_encodings =  tokenizer(list(test_reviews['reviews'].values),padding=True,truncation=True,return_tensors="pt").to(device)


In [33]:
type(train_encodings)

transformers.tokenization_utils_base.BatchEncoding

In [34]:
train_encodings.to(device)

{'input_ids': tensor([[  101,  2074, 12476,  ...,     0,     0,     0],
        [  101,  2204,  2204,  ...,     0,     0,     0],
        [  101,  2204,  2204,  ...,     0,     0,     0],
        ...,
        [  101,  1996,  2190,  ...,     0,     0,     0],
        [  101,  6429,  4031,  ...,     0,     0,     0],
        [  101,  2307,  2007,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [35]:
train_encodings['input_ids']

tensor([[  101,  2074, 12476,  ...,     0,     0,     0],
        [  101,  2204,  2204,  ...,     0,     0,     0],
        [  101,  2204,  2204,  ...,     0,     0,     0],
        ...,
        [  101,  1996,  2190,  ...,     0,     0,     0],
        [  101,  6429,  4031,  ...,     0,     0,     0],
        [  101,  2307,  2007,  ...,     0,     0,     0]])

In [36]:
print(train_encodings['attention_mask'][0].device)
train_encodings['input_ids'][0]

cpu


tensor([  101,  2074, 12476,  1045,  2293,  2023,  2201,  1012,  2009,  1005,
         1055,  2200,  2204,  1012,  2062,  2000,  1996,  5099,  6154,  2217,
         2084,  2014,  2783,  3769,  2614,  1012,  1012,  2061,  1044, 18863,
          999,  1045,  4952,  2000,  2023, 10126,  2012,  1996,  9726,   999,
         1045,  2507,  2009,  1019, 14117,  5790,  2035,  1996,  2126,  1012,
         2014, 19240,  2015,  2024,  2074,  4689,  1012,   102,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

In [37]:
# tokenizer(x,padding=True,truncation=True)['input_ids']

In [38]:
# type(train_encodings['input_ids'])

In [39]:
class ReviewDataSet(torch.utils.data.Dataset):
    def __init__(self,encodings,sentiment):
        self.encodings = encodings
        self.sentiment = sentiment

    def __getitem__(self, index):
        item = {key:val[index] for key,val in self.encodings.items()}
        item['sentiment']=torch.tensor(self.sentiment[index]).to(device)

        return item

    def __len__(self):
        return len(self.sentiment)



In [40]:
train_df = ReviewDataSet(train_encodings,list(train_reviews['user_sentiment'].values))
valid_df = ReviewDataSet(valid_encodings,list(validation_reviews['user_sentiment'].values))
test_df = ReviewDataSet(test_encodings,list(test_reviews['user_sentiment'].values))


In [41]:
train_df[0]['input_ids'].device

device(type='cpu')

In [42]:
from torch.utils.data import DataLoader

In [43]:
len(train_df)


372

In [44]:
torch.accelerator.get_device_capability()

{'supported_dtypes': {torch.bfloat16,
  torch.bool,
  torch.complex32,
  torch.complex64,
  torch.float16,
  torch.float32,
  torch.int16,
  torch.int32,
  torch.int64,
  torch.int8,
  torch.uint16,
  torch.uint32,
  torch.uint64,
  torch.uint8}}

In [45]:
device = 'mps'

In [46]:
BATCH_SIZE = 32
train_loader = DataLoader(train_df,batch_size=BATCH_SIZE,shuffle=True)

In [47]:
val_loader = DataLoader(valid_df,batch_size=BATCH_SIZE,shuffle=False)

In [48]:
test_loader = DataLoader(test_df,batch_size=BATCH_SIZE,shuffle=False)

In [49]:
model

DistilBertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [50]:
len(train_loader)

12

In [51]:
len(train_df)

372

In [52]:
from transformers import DistilBertForSequenceClassification

In [53]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [54]:
model=model.to(device)

In [55]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [56]:
dir(model)

['T_destination',
 '__annotate_func__',
 '__annotations__',
 '__annotations_cache__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_adjust_bias',
 '_adjust_missing_and_unexpected_keys',
 '_apply',
 '_auto_class',
 '_backward_compatibility_gradient_checkpointing',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_can_compile_fullgraph',
 '_can_record_outputs',
 '_can_set_attn_implementation',
 '_can_set_experts_implementation',
 '_can_set_experts_implementation_cached_value',
 '_check_and_adjust_attn_implementation',
 '_check_and_adju

In [57]:
for p,m in model.named_parameters():
    print(p,m.shape)

distilbert.embeddings.word_embeddings.weight torch.Size([30522, 768])
distilbert.embeddings.position_embeddings.weight torch.Size([512, 768])
distilbert.embeddings.LayerNorm.weight torch.Size([768])
distilbert.embeddings.LayerNorm.bias torch.Size([768])
distilbert.transformer.layer.0.attention.q_lin.weight torch.Size([768, 768])
distilbert.transformer.layer.0.attention.q_lin.bias torch.Size([768])
distilbert.transformer.layer.0.attention.k_lin.weight torch.Size([768, 768])
distilbert.transformer.layer.0.attention.k_lin.bias torch.Size([768])
distilbert.transformer.layer.0.attention.v_lin.weight torch.Size([768, 768])
distilbert.transformer.layer.0.attention.v_lin.bias torch.Size([768])
distilbert.transformer.layer.0.attention.out_lin.weight torch.Size([768, 768])
distilbert.transformer.layer.0.attention.out_lin.bias torch.Size([768])
distilbert.transformer.layer.0.sa_layer_norm.weight torch.Size([768])
distilbert.transformer.layer.0.sa_layer_norm.bias torch.Size([768])
distilbert.trans

In [58]:
model.train()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [59]:
optim = torch.optim.AdamW(model.parameters(),lr=.003)

In [60]:
x = torch.tensor([1])

In [61]:
torch.manual_seed(RANDOM_SEED)

In [103]:
torch.mps.seed()

In [104]:
torch.backends.cudnn.deterministic=True

torch.backends.cudnn.benchmark = True

In [105]:
torch.use_deterministic_algorithms(True)

In [106]:
import random
random.seed(RANDOM_SEED)
np.random.seed(0)

In [107]:
#torch.manual_seed(0)
torch.argmax(torch.randn(4,4),dim=1).to(device)

tensor([0, 2, 0, 1], device='mps:0')

In [108]:
device

'mps'

In [109]:
@torch.compile()
def compute_accuracy(model, data_loader, device):
    print(model)
    with torch.no_grad():
        correct_pred, num_examples = 0, 0

        for batch_idx, batch in enumerate(data_loader):
            ### Prepare data
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["sentiment"].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs["logits"]
            predicted_labels = torch.argmax(logits, 1)
            num_examples += labels.size(0)
            correct_pred += (predicted_labels == labels).sum()
            print(f"{num_examples=},{correct_pred=}")
        return correct_pred.float() / num_examples * 100

In [110]:
x = next(iter(train_loader))

In [111]:
x['input_ids']

tensor([[ 101, 2187, 2026,  ...,    0,    0,    0],
        [ 101, 3492, 2204,  ...,    0,    0,    0],
        [ 101, 2307, 2005,  ...,    0,    0,    0],
        ...,
        [ 101, 6919, 2005,  ...,    0,    0,    0],
        [ 101, 2065, 2017,  ...,    0,    0,    0],
        [ 101, 2200, 4138,  ...,    0,    0,    0]])

In [112]:
# import os
# os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO']='0.0'
# os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"


In [113]:
len(train_loader)

12

In [73]:
import time
start = time.time()
from tqdm import tqdm

for epoch in range(1):
    model.train()
    for index,data in enumerate(tqdm(train_loader,desc=f"Training at {epoch}")):

        ### Prepare Data
        input_ids,attention_mask = data['input_ids'],data['attention_mask']
        sentiment = data['sentiment'].to(device)
        #print(input_ids,attention_mask)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        #print(sentiment)
        ### Forward Pass
        outputs = model(input_ids,attention_mask=attention_mask,labels=sentiment)
        #print("Hi")
        loss,logits = outputs['loss'],outputs['logits']
        #print(loss)
        ### Backward Pass
        optim.zero_grad()
        loss.backward()
        optim.step()
        #print("Bye")
        ### Logging
        print(index)
        if not index%250:
            print(
                    f"Epoch: {epoch + 1:04d}/{NUM_EPOCHS:04d} | "
                    f"Batch {index:04d}/{len(train_loader):04d} | "
                    f"Loss: {loss:.4f}"
                )
    model.eval()

    with torch.set_grad_enabled(False):
            #print("Fuck")
            print(
                f"Training accuracy: "
                f"{compute_accuracy(model, train_loader, device):.2f}%"
                f"\nValid accuracy: "
                f"{compute_accuracy(model, val_loader, device):.2f}%"
            )

    print(f"Time elapsed: {(time.time() - start) / 60:.2f} min")


print(f"Total Training Time: {(time.time() - start) / 60:.2f} min")


Training at 0:   0%|          | 0/12 [00:00<?, ?it/s]

0


Training at 0:   8%|▊         | 1/12 [00:01<00:21,  1.94s/it]

Epoch: 0001/0003 | Batch 0000/0012 | Loss: 0.7023
1


Training at 0:  25%|██▌       | 3/12 [00:02<00:07,  1.18it/s]

...

Training at 0: 100%|██████████| 12/12 [00:10<00:00,  1.11it/s]

11


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.14/site-packages/torch/_dynamo/utils.py:4415: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return node.target(*args, **kwargs)  # type: ignore[operator]
/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.14/site-packages/torch/fx/interpreter.py:377: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return target(*args, **kwargs)
/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.14/site-packages/torch/fx/experimental/proxy_tensor.py:2014: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rath

num_examples=32,correct_pred=tensor(30, device='mps:0')
num_examples=64,correct_pred=tensor(59, device='mps:0')
num_examples=96,correct_pred=tensor(83, device='mps:0')
num_examples=128,correct_pred=tensor(110, device='mps:0')
num_examples=160,correct_pred=tensor(140, device='mps:0')
num_examples=192,correct_pred=tensor(168, device='mps:0')
num_examples=224,correct_pred=tensor(192, device='mps:0')
num_examples=256,correct_pred=tensor(217, device='mps:0')
num_examples=288,correct_pred=tensor(246, device='mps:0')
num_examples=320,correct_pred=tensor(270, device='mps:0')
num_examples=352,correct_pred=tensor(295, device='mps:0')
num_examples=372,correct_pred=tensor(310, device='mps:0')
DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout)

In [74]:
print(f"Test accuracy: {compute_accuracy(model, test_loader, device):.2f}%")

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [75]:
del model

In [76]:
import gc
gc.collect()

101905

In [77]:
torch.mps.driver_allocated_memory()

5788483584

In [78]:
torch.mps.empty_cache()

In [79]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [80]:
device = "mps"

In [81]:
model=model.to(device)

In [82]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [83]:
model.train()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [84]:
optim = torch.optim.Adam(model.parameters(),lr=1e-05)

In [85]:


from transformers import Trainer,TrainingArguments

In [86]:
import evaluate

metric = evaluate.load("accuracy")

In [87]:
result = metric.compute(
    predictions=[0, 1, 1],
    references=[0, 1, 0]
)

In [88]:
result

{'accuracy': 0.6666666666666666}

In [97]:
def compute_metric(eval_pred):
    print(eval_pred)
    logits,labels = eval_pred
    predictions = np.argmax(logits,axis=-1)
    return metric.compute(predictions=predictions,references=labels)


In [98]:
training_args = TrainingArguments(
    output_dir='./results/',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    logging_steps=10,

)

In [99]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_df,
    eval_dataset=valid_df,
    compute_metrics=compute_metric,
    optimizers=(optim,None)  # optim and learning rate scheduler
)

In [100]:
start = time.time()

In [101]:
dir(trainer)

['__backends',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_build_accelerator_args',
 '_clip_grad_norm',
 '_created_lr_scheduler',
 '_determine_best_metric',
 '_evaluate',
 '_finalize_training',
 '_finish_current_push',
 '_get_collator_with_removed_columns',
 '_get_dataloader',
 '_get_eval_sampler',
 '_get_grad_norm',
 '_get_learning_rate',
 '_get_num_items_in_batch',
 '_get_output_dir',
 '_get_train_sampler',
 '_has_mixed_mesh_grads',
 '_hp_search_setup',
 '_init_training_state',
 '_inner_training_loop',
 '_issue_warnings_after_load',
 '_load_best_model',
 '_load_callback_state',
 '_load_from_checkpoi

In [102]:
trainer.train()

/Users/deven/.virtualenvs/Machine_Learning_Algorithms/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.